# Retirada das Instâncias viciadas

In [12]:
import pandas as pd

df = pd.read_csv("resultados/instancias_NORM_e_anormal_juntos.csv")

print(f"Quantidade de registros: {len(df)}")

Quantidade de registros: 445


# Divisão dos Rotulos 0 e 1 com subcategorias

Divisão dos Rotulos 0 e 1 com subcategorias aqui , só que use essas informações da instancias_NORM_e_anormal_juntos.csv como critérios para não serem selecionadas para nenhum lugar 


In [13]:

import pandas as pd
import ast
from pathlib import Path

# ============================================================
# 1. CARREGAMENTO DAS BASES
# ============================================================

PASTA = Path.cwd()

def localizar(nome):
    arquivos = sorted(
        p for p in PASTA.rglob("*.csv")
        if p.name.lower().startswith(nome.lower())
        and "resultados" not in p.parts
    )

    if len(arquivos) != 1:
        raise FileNotFoundError(
            f"Esperado um arquivo {nome}, encontrados: {arquivos}"
        )

    return arquivos[0]


df = pd.read_csv(localizar("ptbxl_database"))

scp = pd.read_csv(localizar("scp_statements"))
scp = scp.set_index(scp.columns[0])

excluidos = pd.read_csv(
    PASTA / "resultados" /
    "instancias_NORM_e_anormal_juntos.csv"
)

# ============================================================
# 2. EXCLUSÃO DAS INSTÂNCIAS COM NORM + ANORMALIDADE
# ============================================================

ids_excluidos = set(excluidos["ecg_id"])

df_filtrado = df[
    ~df["ecg_id"].isin(ids_excluidos)
].copy()

print("Registros originais:", len(df))
print("Registros excluídos:", len(df) - len(df_filtrado))
print("Registros restantes:", len(df_filtrado))

# ============================================================
# 3. DEFINIÇÃO DAS SUBCATEGORIAS
# ============================================================

subcategorias = {
    "1.1": "MI",
    "1.2": "STTC",
    "1.3": "CD",
    "1.4": "HYP",
    "1.5": "FORM",
    "1.6": "RHYTHM"
}

# Códigos aceitos como normais, na ausência de alterações
codigos_sinusais = {"SR"}

# Códigos que NÃO devem pertencer a nenhuma classe
CODIGOS_LIMBO = {
    "SBRAD",
    "STACH",
    "SARRH"
}

DESCRICOES_LIMBO = {
    "SBRAD": "Bradicardia sinusal",
    "STACH": "Taquicardia sinusal",
    "SARRH": "Arritmia sinusal"
}

# ============================================================
# 4. CLASSIFICAÇÃO
# ============================================================

def classificar(valor):

    codigos = ast.literal_eval(valor)

    if not isinstance(codigos, dict):
        raise ValueError(
            f"Formato inválido de scp_codes: {valor}"
        )

    # --------------------------------------------------------
    # PRIORIDADE 1: ENVIAR OS TRÊS CÓDIGOS PARA O LIMBO
    # --------------------------------------------------------

    encontrados_limbo = (
        set(codigos.keys()) & CODIGOS_LIMBO
    )

    if encontrados_limbo:
        return pd.Series([
            pd.NA,
            "Limbo",
            "; ".join(sorted(encontrados_limbo))
        ])

    # --------------------------------------------------------
    # PRIORIDADE 2: IDENTIFICAR CATEGORIAS ANORMAIS
    # --------------------------------------------------------

    categorias = set()

    for codigo in codigos:

        if codigo not in scp.index:
            continue

        info = scp.loc[codigo]

        if info["diagnostic"] == 1:

            classe = info["diagnostic_class"]

            for numero, nome in subcategorias.items():

                if classe == nome:
                    categorias.add(numero)

        if (
            info["form"] == 1
            and info["diagnostic"] != 1
        ):
            categorias.add("1.5")

        if (
            info["rhythm"] == 1
            and codigo not in codigos_sinusais
        ):
            categorias.add("1.6")

    if categorias:
        return pd.Series([
            1,
            "; ".join(sorted(categorias)),
            ""
        ])

    # --------------------------------------------------------
    # PRIORIDADE 3: IDENTIFICAR ECGs NORMAIS
    # --------------------------------------------------------

    if (
        "NORM" in codigos
        or "SR" in codigos
    ):
        return pd.Series([
            0,
            "0",
            ""
        ])

    # --------------------------------------------------------
    # PRIORIDADE 4: REGISTROS SEM CLASSIFICAÇÃO
    # --------------------------------------------------------

    return pd.Series([
        pd.NA,
        "Revisar",
        ""
    ])


df_filtrado[
    ["rotulo", "subcategorias", "codigos_limbo"]
] = df_filtrado["scp_codes"].apply(classificar)

# ============================================================
# 5. SEPARAÇÃO DAS CLASSES
# ============================================================

classe_0 = df_filtrado[
    df_filtrado["rotulo"] == 0
].copy()

classe_1 = df_filtrado[
    df_filtrado["rotulo"] == 1
].copy()

# Todos os registros sem classificação
revisar = df_filtrado[
    df_filtrado["rotulo"].isna()
].copy()

# Apenas os registros excluídos pelos três códigos
limbo = df_filtrado[
    df_filtrado["subcategorias"] == "Limbo"
].copy()

# Outros registros que precisam de revisão
revisar_outros = df_filtrado[
    df_filtrado["subcategorias"] == "Revisar"
].copy()

# ============================================================
# 6. RELATÓRIO
# ============================================================

print("\n===== DIVISÃO DOS RÓTULOS =====")

print("Classe 0 (Normal/Sinusal):", len(classe_0))
print("Classe 1 (Anormal):", len(classe_1))
print("Sem classificação (total):", len(revisar))

print("\n===== REGISTROS ENVIADOS AO LIMBO =====")

print("Total no limbo:", len(limbo))

for codigo, descricao in DESCRICOES_LIMBO.items():

    quantidade = limbo["codigos_limbo"].apply(
        lambda valor: codigo in valor.split("; ")
    ).sum()

    print(f"{codigo} - {descricao}: {quantidade}")

print(
    "Outros registros para revisão:",
    len(revisar_outros)
)

print("\n===== SUBCATEGORIAS ANORMAIS =====")

for numero, nome in subcategorias.items():

    quantidade = classe_1["subcategorias"].apply(
        lambda x: numero in x.split("; ")
    ).sum()

    print(f"{numero} - {nome}: {quantidade}")

# ============================================================
# 7. EXPORTAÇÃO DOS REGISTROS DO LIMBO
# ============================================================

PASTA_SAIDA = PASTA / "resultados"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

arquivo_limbo = (
    PASTA_SAIDA /
    "instancias_limbo_SBRAD_STACH_SARRH.csv"
)

limbo.to_csv(
    arquivo_limbo,
    index=False,
    encoding="utf-8-sig"
)

print("\n===== EXPORTAÇÃO DO LIMBO =====")
print("Arquivo salvo em:", arquivo_limbo)

# ============================================================
# 8. VALIDAÇÃO DAS EXCLUSÕES
# ============================================================

assert not set(df_filtrado["ecg_id"]) & ids_excluidos

assert not set(classe_0["ecg_id"]) & ids_excluidos

assert not set(classe_1["ecg_id"]) & ids_excluidos

# Nenhum ECG do limbo pode estar na classe 0
assert not set(limbo["ecg_id"]) & set(
    classe_0["ecg_id"]
)

# Nenhum ECG do limbo pode estar na classe 1
assert not set(limbo["ecg_id"]) & set(
    classe_1["ecg_id"]
)

# Todos os ECGs do limbo devem estar sem classificação
assert limbo["rotulo"].isna().all()

# Nenhum ECG classificado pode conter os códigos proibidos
for conjunto in [classe_0, classe_1]:

    for valor in conjunto["scp_codes"]:

        codigos = ast.literal_eval(valor)

        assert not (
            set(codigos) & CODIGOS_LIMBO
        )

# Conferir a distribuição total
assert (
    len(classe_0)
    + len(classe_1)
    + len(revisar)
    == len(df_filtrado)
)

print("\nValidação concluída:")
print("- Instâncias NORM + anormalidade preservadas na exclusão.")
print("- SBRAD, STACH e SARRH não foram classificados.")
print("- Nenhum registro do limbo pertence às classes 0 ou 1.")
print("- Todos os registros restantes foram contabilizados.")


Registros originais: 21799
Registros excluídos: 445
Registros restantes: 21354

===== DIVISÃO DOS RÓTULOS =====
Classe 0 (Normal/Sinusal): 7257
Classe 1 (Anormal): 11938
Sem classificação (total): 2159

===== REGISTROS ENVIADOS AO LIMBO =====
Total no limbo: 2159
SBRAD - Bradicardia sinusal: 611
STACH - Taquicardia sinusal: 813
SARRH - Arritmia sinusal: 738
Outros registros para revisão: 0

===== SUBCATEGORIAS ANORMAIS =====
1.1 - MI: 4901
1.2 - STTC: 4699
1.3 - CD: 4019
1.4 - HYP: 2340
1.5 - FORM: 6446
1.6 - RHYTHM: 2108

===== EXPORTAÇÃO DO LIMBO =====
Arquivo salvo em: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\instancias_limbo_SBRAD_STACH_SARRH.csv

Validação concluída:
- Instâncias NORM + anormalidade preservadas na exclusão.
- SBRAD, STACH e SARRH não foram classificados.
- Nenhum registro do limbo pertence às classes 0 ou 1.
- Todos os registros restantes foram contabilizados.


In [14]:

import ast
import pandas as pd
from pathlib import Path

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

PASTA = Path.cwd()
SAIDA = PASTA / "resultados" / "Classificacao"

SUBCATEGORIAS = {
    "1.1": "MI",
    "1.2": "STTC",
    "1.3": "CD",
    "1.4": "HYP",
    "1.5": "FORM",
    "1.6": "RHYTHM"
}

SINUSAIS = {"SR"}

# ============================================================
# 2. LOCALIZAÇÃO DOS ARQUIVOS
# ============================================================

def localizar_csv(prefixo, excluir_resultados=False):

    arquivos = sorted(
        p for p in PASTA.rglob("*.csv")
        if p.name.lower().startswith(prefixo.lower())
        and (
            not excluir_resultados
            or "resultados" not in p.relative_to(PASTA).parts
        )
    )

    if not arquivos:
        raise FileNotFoundError(
            f"Arquivo não encontrado: {prefixo}"
        )

    if len(arquivos) > 1:
        raise RuntimeError(
            f"Mais de um arquivo encontrado para {prefixo}:\n"
            + "\n".join(map(str, arquivos))
            + "\nInforme a versão desejada para evitar ambiguidades."
        )

    return arquivos[0]


ARQUIVO_ECG = localizar_csv(
    "ptbxl_database",
    excluir_resultados=True
)

ARQUIVO_SCP = localizar_csv(
    "scp_statements",
    excluir_resultados=True
)

# Procura primeiro o CSV de exclusões na pasta resultados.
# Aceita também nomes como (...)(1).csv.

pasta_exclusoes = PASTA / "resultados"

candidatos = sorted(
    p for p in pasta_exclusoes.glob(
        "instancias_NORM_e_anormal_juntos*.csv"
    )
    if p.is_file()
)

if len(candidatos) == 1:
    ARQUIVO_EXCLUSOES = candidatos[0]

elif len(candidatos) == 0:
    ARQUIVO_EXCLUSOES = localizar_csv(
        "instancias_NORM_e_anormal_juntos"
    )

else:
    raise RuntimeError(
        "Há vários CSVs de exclusão. "
        "Mantenha somente a versão desejada:\n"
        + "\n".join(map(str, candidatos))
    )

print("Base PTB-XL:", ARQUIVO_ECG)
print("Dicionário SCP:", ARQUIVO_SCP)
print("Tabela de exclusão:", ARQUIVO_EXCLUSOES)

# ============================================================
# 3. CARREGAMENTO
# ============================================================

df_original = pd.read_csv(ARQUIVO_ECG)
scp = pd.read_csv(ARQUIVO_SCP)
exclusoes = pd.read_csv(ARQUIVO_EXCLUSOES)

scp = scp.set_index(scp.columns[0])

# ============================================================
# 4. VALIDAÇÃO DA TABELA DE EXCLUSÃO
# ============================================================

if "ecg_id" not in exclusoes.columns:
    raise ValueError(
        "A tabela de exclusão não possui a coluna ecg_id."
    )

if exclusoes["ecg_id"].isna().any():
    raise ValueError(
        "Existem ecg_id vazios na tabela de exclusão."
    )

if exclusoes["ecg_id"].duplicated().any():
    raise ValueError(
        "Existem ecg_id duplicados na tabela de exclusão."
    )

ids_excluidos = set(
    exclusoes["ecg_id"].astype(int)
)

ids_originais = set(
    df_original["ecg_id"].astype(int)
)

if len(ids_excluidos) != 445:
    raise ValueError(
        "Esperados 445 IDs na tabela de exclusão, "
        f"mas foram encontrados {len(ids_excluidos)}."
    )

if not ids_excluidos.issubset(ids_originais):
    faltantes = ids_excluidos - ids_originais

    raise ValueError(
        "Existem IDs de exclusão ausentes na base: "
        f"{sorted(faltantes)}"
    )

print("\n===== EXCLUSÕES =====")
print("Total original:", len(df_original))
print("IDs de exclusão:", len(ids_excluidos))

# ============================================================
# 5. EXCLUSÃO ANTES DA CLASSIFICAÇÃO
# ============================================================

mascara_exclusao = df_original[
    "ecg_id"
].astype(int).isin(ids_excluidos)

df = df_original.loc[
    ~mascara_exclusao
].copy()

removidos = df_original.loc[
    mascara_exclusao
].copy()

assert len(removidos) == 445
assert not set(df["ecg_id"]) & ids_excluidos

print("Instâncias removidas:", len(removidos))
print("Instâncias restantes:", len(df))

# ============================================================
# 6. INTERPRETAÇÃO DOS CÓDIGOS
# ============================================================

def interpretar(valor):

    if isinstance(valor, dict):
        return valor

    codigos = ast.literal_eval(valor)

    if not isinstance(codigos, dict):
        raise ValueError(
            f"scp_codes inválido: {valor}"
        )

    return codigos


# ============================================================
# 7. CLASSIFICAÇÃO
# ============================================================

def classificar(valor):

    codigos = interpretar(valor)

    categorias = set()
    desconhecidos = []

    for codigo in codigos:

        if codigo not in scp.index:
            desconhecidos.append(codigo)
            continue

        info = scp.loc[codigo]

        # Superclasses diagnósticas
        if info["diagnostic"] == 1:

            classe = info["diagnostic_class"]

            for numero, nome in SUBCATEGORIAS.items():

                if classe == nome:
                    categorias.add(numero)

        # Alterações morfológicas
        if (
            info["form"] == 1
            and info["diagnostic"] != 1
        ):
            categorias.add("1.5")

        # Alterações de ritmo não sinusais
        if (
            info["rhythm"] == 1
            and codigo not in SINUSAIS
        ):
            categorias.add("1.6")

    if desconhecidos:
        return pd.Series([
            pd.NA,
            "",
            "REVISAR_CODIGO_DESCONHECIDO"
        ])

    # Alteração prevalece sobre normalidade/sinusal
    if categorias:
        return pd.Series([
            1,
            "; ".join(sorted(categorias)),
            "ANORMAL"
        ])

    # Normal ou sinusal
    if (
        "NORM" in codigos
        or bool(set(codigos) & SINUSAIS)
    ):
        return pd.Series([
            0,
            "0",
            "NORMAL_SINUSAL"
        ])

    return pd.Series([
        pd.NA,
        "",
        "REVISAR"
    ])


df[["rotulo", "subcategorias", "situacao"]] = (
    df["scp_codes"].apply(classificar)
)

# ============================================================
# 8. SEPARAÇÃO DAS CLASSES
# ============================================================

classe_0 = df[
    df["rotulo"] == 0
].copy()

classe_1 = df[
    df["rotulo"] == 1
].copy()

revisar = df[
    df["rotulo"].isna()
].copy()

# ============================================================
# 9. PREPARAR SUBCATEGORIAS
# ============================================================

dados_subcategorias = {}
contagens = {}

for numero, nome in SUBCATEGORIAS.items():

    mascara = classe_1["subcategorias"].apply(
        lambda valor: numero in valor.split("; ")
    )

    dados_sub = classe_1.loc[mascara].copy()

    dados_subcategorias[numero] = dados_sub
    contagens[numero] = len(dados_sub)

# ============================================================
# 10. VALIDAÇÃO OBRIGATÓRIA
# ============================================================

def validar_sem_excluidos(nome, dados):

    encontrados = set(
        dados["ecg_id"].astype(int)
    ) & ids_excluidos

    if encontrados:
        raise AssertionError(
            f"ERRO: {nome} contém IDs excluídos: "
            f"{sorted(encontrados)}"
        )


validar_sem_excluidos("Base filtrada", df)
validar_sem_excluidos("Classe 0", classe_0)
validar_sem_excluidos("Classe 1", classe_1)
validar_sem_excluidos("Revisar", revisar)

for numero, dados_sub in dados_subcategorias.items():

    validar_sem_excluidos(
        f"Subcategoria {numero}",
        dados_sub
    )

assert len(df_original) == len(df) + len(removidos)

assert (
    len(classe_0)
    + len(classe_1)
    + len(revisar)
    == len(df)
)

assert not (
    set(classe_0["ecg_id"])
    & set(classe_1["ecg_id"])
)

print("\nVALIDAÇÃO APROVADA")
print("Nenhuma das 445 instâncias entrou nas classes.")

# ============================================================
# 11. EXPORTAÇÃO DOS CSVs
# ============================================================

PASTA_0 = SAIDA / "0"
PASTA_1 = SAIDA / "1"
PASTA_REVISAO = SAIDA / "Revisar"

for pasta in [PASTA_0, PASTA_1, PASTA_REVISAO]:
    pasta.mkdir(parents=True, exist_ok=True)

# Sobrescreve os CSVs de classificação anteriores
classe_0.to_csv(
    PASTA_0 / "classe_0.csv",
    index=False,
    encoding="utf-8-sig"
)

classe_1.to_csv(
    PASTA_1 / "classe_1.csv",
    index=False,
    encoding="utf-8-sig"
)

revisar.to_csv(
    PASTA_REVISAO / "revisar.csv",
    index=False,
    encoding="utf-8-sig"
)

for numero, nome in SUBCATEGORIAS.items():

    pasta_sub = PASTA_1 / numero
    pasta_sub.mkdir(parents=True, exist_ok=True)

    dados_subcategorias[numero].to_csv(
        pasta_sub / f"classe_{numero}_{nome}.csv",
        index=False,
        encoding="utf-8-sig"
    )

# ============================================================
# 12. RELATÓRIO FINAL
# ============================================================

print("\n" + "=" * 55)
print("RELATÓRIO FINAL")
print("=" * 55)

print("Total original:", len(df_original))
print("Instâncias excluídas:", len(removidos))
print("Total após exclusões:", len(df))

print("\nCLASSES")

print("0 - Normal/Sinusal:", len(classe_0))
print("1 - Anormal:", len(classe_1))
print("Revisar:", len(revisar))

print("\nSUBCATEGORIAS")

for numero, nome in SUBCATEGORIAS.items():

    print(
        f"{numero} - {nome}: "
        f"{contagens[numero]}"
    )

print("\n" + "=" * 55)
print("EXCLUSÃO VERIFICADA COM SUCESSO")
print("=" * 55)

print("445 instâncias excluídas.")
print("0 instâncias excluídas na classe 0.")
print("0 instâncias excluídas na classe 1.")
print("0 instâncias excluídas nas subcategorias.")

print(f"\nArquivos salvos em: {SAIDA}")


Base PTB-XL: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\PTB-XL\ptbxl_database.csv
Dicionário SCP: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\PTB-XL\scp_statements.csv
Tabela de exclusão: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\instancias_NORM_e_anormal_juntos.csv

===== EXCLUSÕES =====
Total original: 21799
IDs de exclusão: 445
Instâncias removidas: 445
Instâncias restantes: 21354

VALIDAÇÃO APROVADA
Nenhuma das 445 instâncias entrou nas classes.

RELATÓRIO FINAL
Total original: 21799
Instâncias excluídas: 445
Total após exclusões: 21354

CLASSES
0 - Normal/Sinusal: 7257
1 - Anormal: 14097
Revisar: 0

SUBCATEGORIAS
1.1 - MI: 5468
1.2 - STTC: 5202
1.3 - CD: 4483
1.4 - HYP: 2644
1.5 - FORM: 7233
1.6 - RHYTHM: 4267

EXCLUSÃO VERIFICADA COM SUCESSO
445 instâncias excluídas.
0 instâncias excluídas na classe 0.
0 instâncias excluídas na classe 1.
0 instâncias excluídas nas subcategorias.

Arquivos salvos em: c:\Users\Matheus\Documents\Detctor-automati

# Análise da Quantidade de Instâncias Únicas por Classe e Subcategoria

In [15]:
from pathlib import Path
import pandas as pd

PASTA = Path("resultados/Classificacao")

arquivos = {
    "0 - Normal/Sinusal": PASTA / "0/classe_0.csv",
    "1 - Anormal": PASTA / "1/classe_1.csv",
    "1.1 - MI": PASTA / "1/1.1/classe_1.1_MI.csv",
    "1.2 - STTC": PASTA / "1/1.2/classe_1.2_STTC.csv",
    "1.3 - CD": PASTA / "1/1.3/classe_1.3_CD.csv",
    "1.4 - HYP": PASTA / "1/1.4/classe_1.4_HYP.csv",
    "1.5 - FORM": PASTA / "1/1.5/classe_1.5_FORM.csv",
    "1.6 - RHYTHM": PASTA / "1/1.6/classe_1.6_RHYTHM.csv"
}

resultado = []

for classe, caminho in arquivos.items():
    df_temp = pd.read_csv(caminho)

    resultado.append({
        "Classe": classe,
        "Total": len(df_temp),
        "Únicos (ecg_id)": df_temp["ecg_id"].nunique(),
        "Duplicados": df_temp["ecg_id"].duplicated().sum()
    })

resumo = pd.DataFrame(resultado)

display(resumo)

,Classe,Total,Únicos (ecg_id),Duplicados
0,0 - Normal/Sinusal,7257,7257,0
1,1 - Anormal,14097,14097,0
2,1.1 - MI,5468,5468,0
3,1.2 - STTC,5202,5202,0
4,1.3 - CD,4483,4483,0
5,1.4 - HYP,2644,2644,0
6,1.5 - FORM,7233,7233,0
7,1.6 - RHYTHM,4267,4267,0


In [16]:
import pandas as pd

df_mi = pd.read_csv(
    "resultados/Classificacao/1/1.1/classe_1.1_MI.csv"
)

colunas = [
    "ecg_id", "patient_id", "age", "sex", "height", "weight",
    "nurse", "site", "device", "recording_date", "report",
    "scp_codes", "heart_axis", "infarction_stadium1",
    "infarction_stadium2", "validated_by", "second_opinion",
    "initial_autogenerated_report", "validated_by_human",
    "baseline_drift", "static_noise", "burst_noise",
    "electrodes_problems", "extra_beats", "pacemaker",
    "strat_fold", "filename_lr", "filename_hr",
    "rotulo", "subcategorias", "situacao"
]

display(df_mi[colunas])

,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,rotulo,subcategorias,situacao
0,8,11275.0,48.0,0,NaN,95.0,2.0,0.0,CS-12 E,1984-12-01 14:49:52,...,NaN,NaN,NaN,NaN,9,records100/00000/00008_lr,records500/00000/00008_hr,1,1.1; 1.5,ANORMAL
1,39,13619.0,56.0,0,NaN,NaN,2.0,0.0,CS-12 E,1985-02-16 12:54:01,...,NaN,NaN,NaN,NaN,9,records100/00000/00039_lr,records500/00000/00039_hr,1,1.1; 1.2; 1.5,ANORMAL
2,50,16961.0,59.0,1,NaN,64.0,2.0,0.0,CS-12 E,1985-05-09 12:01:24,...,NaN,NaN,NaN,NaN,7,records100/00000/00050_lr,records500/00000/00050_hr,1,1.1; 1.3,ANORMAL
3,63,15265.0,30.0,0,NaN,78.0,2.0,0.0,CS-12 E,1985-07-03 09:27:10,...,NaN,NaN,NaN,NaN,10,records100/00000/00063_lr,records500/00000/00063_hr,1,1.1; 1.5,ANORMAL
4,77,8555.0,43.0,0,NaN,63.0,0.0,0.0,CS-12 E,1985-09-27 09:55:22,...,NaN,NaN,NaN,NaN,6,records100/00000/00077_lr,records500/00000/00077_hr,1,1.1; 1.3,ANORMAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5463,21820,18655.0,80.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-03-03 17:34:11,...,NaN,NaN,NaN,NaN,7,records100/21000/21820_lr,records500/21000/21820_hr,1,1.1,ANORMAL
5464,21824,19977.0,69.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-05-10 08:10:23,...,NaN,NaN,NaN,NaN,8,records100/21000/21824_lr,records500/21000/21824_hr,1,1.1; 1.5,ANORMAL
5465,21826,9178.0,82.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-13 18:20:52,...,NaN,NaN,NaN,NaN,10,records100/21000/21826_lr,records500/21000/21826_hr,1,1.1; 1.5; 1.6,ANORMAL
5466,21827,13862.0,79.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-20 16:30:53,...,NaN,NaN,"4ES,SVES",NaN,5,records100/21000/21827_lr,records500/21000/21827_hr,1,1.1; 1.2; 1.5; 1.6,ANORMAL


### Alocando os Reespctivos registros nas suas reeespectivas labels e sublabels(ECG-Organizado)

In [17]:
import pandas as pd
import shutil
from pathlib import Path

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

PASTA = Path.cwd()

BASE_PTBXL = PASTA / "PTB-XL"
CLASSIFICACAO = PASTA / "resultados" / "Classificacao"
SAIDA = PASTA / "resultados" / "ECG_Organizados"

SUBCATEGORIAS = {
    "1.1": "MI",
    "1.2": "STTC",
    "1.3": "CD",
    "1.4": "HYP",
    "1.5": "FORM",
    "1.6": "RHYTHM"
}

# ============================================================
# 2. CARREGAR AS 445 INSTÂNCIAS EXCLUÍDAS
# ============================================================

arquivos_exclusao = list(
    (PASTA / "resultados").glob(
        "instancias_NORM_e_anormal_juntos*.csv"
    )
)

if len(arquivos_exclusao) != 1:
    raise RuntimeError(
        "É necessário encontrar exatamente um CSV de exclusões "
        "na pasta resultados."
    )

exclusoes = pd.read_csv(arquivos_exclusao[0])

ids_excluidos = set(
    exclusoes["ecg_id"].astype(int)
)

assert len(ids_excluidos) == 445

# ============================================================
# 3. ORGANIZAR OS REGISTROS
# ============================================================

def organizar(nome_classe, arquivo_csv, destino):

    df = pd.read_csv(arquivo_csv)

    # Garantir que nenhuma instância excluída entre
    ids = set(df["ecg_id"].astype(int))

    conflitantes = ids & ids_excluidos

    if conflitantes:
        raise ValueError(
            f"{nome_classe} contém instâncias excluídas: "
            f"{sorted(conflitantes)}"
        )

    pasta_registros = destino / "registros"
    pasta_registros.mkdir(parents=True, exist_ok=True)

    registros_exportados = []
    ausentes = []

    for _, instancia in df.iterrows():

        ecg_id = int(instancia["ecg_id"])

        # Exemplo:
        # records500/00000/00004_hr
        caminho_relativo = Path(instancia["filename_hr"])

        origem_base = BASE_PTBXL / caminho_relativo

        origem_dat = origem_base.with_suffix(".dat")
        origem_hea = origem_base.with_suffix(".hea")

        if not origem_dat.exists() or not origem_hea.exists():

            ausentes.append(ecg_id)
            continue

        # Mantém a estrutura relativa de diretórios dentro
        # de registros para preservar possíveis referências.
        destino_base = pasta_registros / caminho_relativo

        destino_base.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        destino_dat = destino_base.with_suffix(".dat")
        destino_hea = destino_base.with_suffix(".hea")

        shutil.copy2(origem_dat, destino_dat)
        shutil.copy2(origem_hea, destino_hea)

        registro = instancia.to_dict()

        registro["arquivo_dat"] = str(
            destino_dat.relative_to(destino)
        )

        registro["arquivo_hea"] = str(
            destino_hea.relative_to(destino)
        )

        registros_exportados.append(registro)

    # CSV apenas com os registros efetivamente copiados
    df_saida = pd.DataFrame(
        registros_exportados,
        columns=list(df.columns) + [
            "arquivo_dat",
            "arquivo_hea"
        ]
    )

    df_saida.to_csv(
        destino / "instancias.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"{nome_classe}: "
        f"{len(df_saida)} copiados | "
        f"{len(ausentes)} ausentes"
    )

    return ausentes


# ============================================================
# 4. CLASSE 0
# ============================================================

todos_ausentes = {}

todos_ausentes["0"] = organizar(
    "0 - Normal/Sinusal",
    CLASSIFICACAO / "0" / "classe_0.csv",
    SAIDA / "0"
)

# ============================================================
# 5. SUBCATEGORIAS DA CLASSE 1
# ============================================================

for numero, nome in SUBCATEGORIAS.items():

    arquivo = (
        CLASSIFICACAO
        / "1"
        / numero
        / f"classe_{numero}_{nome}.csv"
    )

    destino = SAIDA / "1" / f"{numero}_{nome}"

    todos_ausentes[numero] = organizar(
        f"{numero} - {nome}",
        arquivo,
        destino
    )

# ============================================================
# 6. RELATÓRIO FINAL
# ============================================================

print("\n" + "=" * 55)
print("ORGANIZAÇÃO CONCLUÍDA")
print("=" * 55)

print("IDs conflitantes bloqueados:", len(ids_excluidos))
print("Pasta de saída:", SAIDA)

total_ausentes = sum(
    len(ids) for ids in todos_ausentes.values()
)

print("Ocorrências de arquivos ausentes:", total_ausentes)

if total_ausentes:
    print(
        "Atenção: alguns registros não foram copiados. "
        "Confira BASE_PTBXL e os caminhos filename_hr."
    )

0 - Normal/Sinusal: 7257 copiados | 0 ausentes
1.1 - MI: 5468 copiados | 0 ausentes
1.2 - STTC: 5202 copiados | 0 ausentes
1.3 - CD: 4483 copiados | 0 ausentes
1.4 - HYP: 2644 copiados | 0 ausentes
1.5 - FORM: 7233 copiados | 0 ausentes
1.6 - RHYTHM: 4267 copiados | 0 ausentes

ORGANIZAÇÃO CONCLUÍDA
IDs conflitantes bloqueados: 445
Pasta de saída: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\ECG_Organizados
Ocorrências de arquivos ausentes: 0


# Arquivo: 2-extracao_qrs_corrigida_todos_menos_445

# Amostragem Representativa e Análise Preliminar dos Complexos QRS por Classe e Derivação

In [8]:
from matplotlib.lines import Line2D

# ============================================================
# 7. VISUALIZAÇÃO DOS QRS QUE SERIAM RECORTADOS
# ============================================================

for _, ecg in df_selecionados.iterrows():

    caminho = BASE / str(ecg["filename_hr"])
    registro = wfdb.rdrecord(str(caminho))

    fs = float(registro.fs)
    sinais = registro.p_signal
    tempo = np.arange(sinais.shape[0]) / fs

    fig, eixos = plt.subplots(
        12, 1,
        figsize=(18, 24),
        sharex=True
    )

    for i, derivacao in enumerate(DERIVACOES):

        ax = eixos[i]

        if derivacao not in registro.sig_name:
            ax.set_title(f"{derivacao} - Ausente")
            continue

        indice = registro.sig_name.index(derivacao)
        sinal = sinais[:, indice]

        picos, filtrado, _ = detectar_qrs(sinal, fs)

        # ----------------------------------------------------
        # A. EXIBIR O SINAL ORIGINAL
        # ----------------------------------------------------

        ax.plot(
            tempo,
            sinal,
            color="gray",
            linewidth=0.9,
            label="Sinal original"
        )

        # ----------------------------------------------------
        # B. IDENTIFICAR OS QRS
        # ----------------------------------------------------

        for pico in picos:

            inicio, fim, situacao = estimar_limites(
                filtrado,
                pico,
                fs
            )

            # Caso não seja possível determinar os limites
            if inicio is None:

                ax.axvline(
                    pico / fs,
                    color="orange",
                    linestyle=":",
                    linewidth=1.2
                )

                continue

            # ------------------------------------------------
            # C. APLICAR A REGRA DOS 120 MS
            # ------------------------------------------------

            duracao = (fim - inicio) / fs * 1000

            if duracao >= 120:
                cor = "red"
            else:
                cor = "green"

            # ------------------------------------------------
            # D. DESTACAR VISUALMENTE O QRS QUE SERIA RECORTADO
            # ------------------------------------------------

            ax.plot(
                tempo[inicio:fim + 1],
                sinal[inicio:fim + 1],
                color=cor,
                linewidth=3.5,
                zorder=5
            )

            # Marcar o início do QRS
            ax.axvline(
                inicio / fs,
                color=cor,
                linestyle="--",
                linewidth=1,
                alpha=0.7
            )

            # Marcar o fim do QRS
            ax.axvline(
                fim / fs,
                color=cor,
                linestyle="--",
                linewidth=1,
                alpha=0.7
            )

            # Destacar o pico R
            ax.scatter(
                tempo[pico],
                sinal[pico],
                color="blue",
                s=18,
                zorder=6
            )

        # ----------------------------------------------------
        # E. CONFIGURAÇÃO DO GRÁFICO DA DERIVAÇÃO
        # ----------------------------------------------------

        ax.set_ylabel(
            derivacao,
            fontweight="bold",
            fontsize=11
        )

        ax.grid(alpha=0.2)

    # ========================================================
    # 8. TÍTULO DO ECG
    # ========================================================

    fig.suptitle(
        f"ECG {ecg['ecg_id']} | "
        f"Classe {ecg['rotulo']} - {ecg['categoria']}\n"
        "Visualização dos segmentos QRS candidatos ao recorte",
        fontsize=15,
        fontweight="bold",
        y=0.995
    )

    # ========================================================
    # 9. LEGENDA
    # ========================================================

    legenda = [
        Line2D(
            [0], [0],
            color="gray",
            lw=1.5,
            label="Sinal original"
        ),

        Line2D(
            [0], [0],
            color="green",
            lw=3,
            label="QRS < 120 ms"
        ),

        Line2D(
            [0], [0],
            color="red",
            lw=3,
            label="QRS ≥ 120 ms"
        ),

        Line2D(
            [0], [0],
            color="blue",
            marker="o",
            linestyle="None",
            markersize=5,
            label="Pico R"
        ),

        Line2D(
            [0], [0],
            color="orange",
            lw=1.5,
            linestyle=":",
            label="Revisar"
        )
    ]

    fig.legend(
        handles=legenda,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.965),
        ncol=5,
        fontsize=10
    )

    # ========================================================
    # 10. EXIBIR O GRÁFICO
    # ========================================================

    eixos[-1].set_xlabel(
        "Tempo (segundos)",
        fontsize=12
    )

    plt.tight_layout(
        rect=[0, 0, 1, 0.94]
    )

    plt.show()
    plt.close(fig)

NameError: name 'df_selecionados' is not defined

In [18]:
from pathlib import Path

PASTA = Path.cwd()

ORIGEM = PASTA / "PTB-XL"
CLASSIFICACAO = PASTA / "resultados" / "Classificacao"
DESTINO = PASTA / "Recortes-QRS"

print("Origem:", ORIGEM)
print("Classificação:", CLASSIFICACAO)
print("Destino:", DESTINO)

Origem: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\PTB-XL
Classificação: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\Classificacao
Destino: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\Recortes-QRS


In [19]:
# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

from pathlib import Path

PASTA = Path.cwd()

# Localizar a pasta ECG_Organizados
pastas_encontradas = [
    pasta
    for pasta in PASTA.rglob("*")
    if pasta.is_dir()
    and pasta.name.lower() == "ecg_organizados"
    and "Recortes-QRS" not in pasta.parts
]

if not pastas_encontradas:
    raise FileNotFoundError(
        f"Pasta ECG_Organizados não encontrada em {PASTA}"
    )

ORIGEM = pastas_encontradas[0]

DESTINO = PASTA / "Recortes-QRS"

DERIVACOES = [
    "I", "II", "III",
    "aVR", "aVL", "aVF",
    "V1", "V2", "V3",
    "V4", "V5", "V6"
]

LIMIAR_QRS_MS = 120

MODO_TESTE = True
AMOSTRAS_TESTE_POR_CLASSE = 2

SOBRESCREVER = False

print("Origem:", ORIGEM)
print("Destino:", DESTINO)

Origem: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\ECG_Organizados
Destino: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\Recortes-QRS


In [20]:
import re
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, sosfiltfilt, find_peaks
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURAÇÕES: TODAS AS AMOSTRAS
# ============================================================
PASTA = Path.cwd()
DESTINO = PASTA / 'Recortes-QRS'
DERIVACOES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF',
              'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
CLASSES = ['0', '1.1_MI', '1.2_STTC', '1.3_CD',
           '1.4_HYP', '1.5_FORM', '1.6_RHYTHM']
LIMIAR_QRS_MS = 120
SOBRESCREVER = False  # True apenas para reprocessar pastas completas
JANELA_ANTES_MS = 120
JANELA_DEPOIS_MS = 160

# ============================================================
# 2. ENCONTRAR ORIGEM SEM MODIFICÁ-LA
# ============================================================
candidatas = [p for p in PASTA.rglob('*')
              if p.is_dir() and p.name.casefold() == 'ecg_organizados'
              and DESTINO.resolve() not in p.resolve().parents]
if len(candidatas) != 1:
    raise RuntimeError(f'Esperada uma pasta ECG_Organizados; encontradas: {candidatas}')
ORIGEM = candidatas[0]
if ORIGEM.resolve() == DESTINO.resolve() or ORIGEM.resolve() in DESTINO.resolve().parents:
    raise RuntimeError('Origem e destino não podem coincidir ou se sobrepor.')

# ============================================================
# 3. EXCLUSÃO DAS 445 INSTÂNCIAS
# ============================================================
arquivos_exclusao = sorted((PASTA / 'resultados').glob('instancias_NORM_e_anormal_juntos*.csv'))
if len(arquivos_exclusao) != 1:
    raise RuntimeError('É necessário exatamente um CSV de exclusões em resultados/.')
exclusoes = pd.read_csv(arquivos_exclusao[0])
if 'ecg_id' not in exclusoes:
    raise ValueError('CSV de exclusões sem coluna ecg_id.')
ids_excluidos = set(pd.to_numeric(exclusoes.ecg_id, errors='raise').astype(int))
if len(ids_excluidos) != 445:
    raise ValueError(f'Esperados 445 IDs únicos excluídos; encontrados {len(ids_excluidos)}.')

# ============================================================
# 4. DETECÇÃO EXPLORATÓRIA DE QRS
# Se as funções já foram definidas no notebook, preservá-las.
# ============================================================
if not callable(globals().get('detectar_qrs')):
    def detectar_qrs(sinal, fs):
        sinal = np.asarray(sinal, dtype=float)
        if len(sinal) < 30 or not np.isfinite(sinal).all():
            return [], None, 'SINAL_INVALIDO'
        filtrado = sosfiltfilt(butter(3, [5, 25], btype='bandpass', fs=fs, output='sos'), sinal)
        energia = filtrado ** 2
        janela = max(1, int(.08 * fs))
        energia_suave = np.convolve(energia, np.ones(janela) / janela, mode='same')
        limiar = np.median(energia_suave) + .5 * np.std(energia_suave)
        candidatos, _ = find_peaks(
            energia_suave, distance=max(1, int(.25 * fs)), height=limiar,
            prominence=max(np.std(energia_suave) * .25, np.finfo(float).eps))
        picos = []
        for candidato in candidatos:
            margem = int(.06 * fs)
            a, b = max(0, candidato - margem), min(len(filtrado), candidato + margem + 1)
            if a < b:
                picos.append(int(a + np.argmax(np.abs(filtrado[a:b]))))
        selecionados = []
        for pico in sorted(set(picos)):
            if not selecionados or pico - selecionados[-1] >= int(.20 * fs):
                selecionados.append(pico)
            elif abs(filtrado[pico]) > abs(filtrado[selecionados[-1]]):
                selecionados[-1] = pico
        return selecionados, filtrado, ('DETECCAO_PRELIMINAR' if selecionados else 'PICO_R_NAO_DETECTADO')

if not callable(globals().get('estimar_limites')):
    def estimar_limites(sinal_filtrado, pico, fs):
        antes, depois = int(JANELA_ANTES_MS * fs / 1000), int(JANELA_DEPOIS_MS * fs / 1000)
        a, b = max(0, pico - antes), min(len(sinal_filtrado) - 1, pico + depois)
        if a == 0 or b == len(sinal_filtrado) - 1:
            return None, None, 'QRS_PROXIMO_DA_BORDA'
        amplitude = np.abs(sinal_filtrado)
        referencia = amplitude[pico]
        if referencia <= 0:
            return None, None, 'AMPLITUDE_INSUFICIENTE'
        limiar = .10 * referencia
        inicio = fim = int(pico)
        while inicio > a and amplitude[inicio] > limiar:
            inicio -= 1
        while fim < b and amplitude[fim] > limiar:
            fim += 1
        if inicio >= pico or fim <= pico:
            return None, None, 'LIMITES_INDEFINIDOS'
        if inicio == a or fim == b:
            return None, None, 'LIMITES_NAO_CONFIAVEIS'
        return inicio, fim, 'LIMITES_ESTIMADOS'

# ============================================================
# 5. IDENTIFICAR AMOSTRAS E PRESERVAR CAMINHOS ORIGINAIS
# ============================================================
def classe_do_caminho(rel):
    partes = rel.parts
    if len(partes) >= 3 and partes[0] == '0' and partes[1].startswith('registros'):
        return '0'
    if len(partes) >= 4 and partes[0] == '1' and partes[1] in CLASSES[1:] and partes[2].startswith('registros'):
        return partes[1]
    return None

registros = []
ignorados = {'excluidos': 0, 'sem_dat': 0, 'fora_estrutura': 0, 'nome_invalido': 0}
for hea in sorted(ORIGEM.rglob('*.hea')):
    rel = hea.parent.relative_to(ORIGEM)
    classe = classe_do_caminho(rel)
    if classe is None:
        ignorados['fora_estrutura'] += 1
        continue
    if not rel.parts[-1].isdigit():
        ignorados['nome_invalido'] += 1
        continue
    ecg_id = int(rel.parts[-1])
    # O ID na pasta deve corresponder ao registro WFDB
    m = re.fullmatch(r'(\d+)(?:_hr|_lr)?', hea.stem)
    if m is None or int(m.group(1)) != ecg_id:
        ignorados['nome_invalido'] += 1
        continue
    if ecg_id in ids_excluidos:
        ignorados['excluidos'] += 1
        continue
    dat = hea.with_suffix('.dat')
    if not dat.is_file():
        ignorados['sem_dat'] += 1
        continue
    registros.append({'id': ecg_id, 'classe': classe, 'hea': hea,
                      'dat': dat, 'rel': rel})

# Evita sobrescrever o mesmo destino caso haja mais de um registro na pasta.
destinos = [str(item['rel']).casefold() for item in registros]
if len(destinos) != len(set(destinos)):
    raise RuntimeError('Há mais de um par WFDB para a mesma pasta de amostra; revise antes de processar.')

print('Origem (somente leitura):', ORIGEM)
print('Destino:', DESTINO)
print('Registros elegíveis:', len(registros))
print('Ignorados:', ignorados)
if not registros:
    raise RuntimeError('Nenhuma amostra elegível encontrada na estrutura esperada.')

# ============================================================
# 6. PROCESSAR E SALVAR CADA AMOSTRA INDIVIDUALMENTE
# ============================================================
COLUNAS_QRS = ['ecg_id', 'derivacao', 'batimento', 'amostra_qrs',
               'amostra_original', 'tempo_relativo_ms', 'amplitude_mV',
               'duracao_qrs_ms', 'qrs_alargado']
COLUNAS_META = ['ecg_id', 'classe', 'derivacao', 'batimento', 'pico_r',
                'inicio_qrs', 'fim_qrs', 'duracao_ms', 'qrs_alargado',
                'situacao', 'motivo', 'fs_hz']

def processar_amostra(item):
    destino = DESTINO / item['rel']
    esperados = [item['dat'].name, item['hea'].name, 'instancias.csv'] + [f'{d}_QRS.csv' for d in DERIVACOES]
    if destino.exists() and not SOBRESCREVER:
        completos = all((destino / nome).is_file() for nome in esperados)
        return {'ecg_id': item['id'], 'classe': item['classe'],
                'situacao': 'JA_PROCESSADO' if completos else 'DESTINO_INCOMPLETO', 'qrs_extraidos': 0}

    registro = wfdb.rdrecord(str(item['hea'].with_suffix('')))
    fs = float(registro.fs)
    if registro.p_signal is None or fs <= 0:
        raise ValueError('Sinal físico ausente ou frequência inválida.')
    metadados = []
    por_derivacao = {}
    total = 0
    for derivacao in DERIVACOES:
        linhas = []
        def registrar(batimento=None, pico=None, inicio=None, fim=None,
                     duracao=None, alargado=None, situacao='REVISAR', motivo=''):
            metadados.append({'ecg_id': item['id'], 'classe': item['classe'],
                              'derivacao': derivacao, 'batimento': batimento,
                              'pico_r': pico, 'inicio_qrs': inicio, 'fim_qrs': fim,
                              'duracao_ms': duracao, 'qrs_alargado': alargado,
                              'situacao': situacao, 'motivo': motivo, 'fs_hz': fs})
        if derivacao not in registro.sig_name:
            registrar(motivo='DERIVACAO_AUSENTE')
        else:
            sinal = registro.p_signal[:, registro.sig_name.index(derivacao)]
            picos, filtrado, status = detectar_qrs(sinal, fs)
            if not picos:
                registrar(motivo=status)
            else:
                for numero, pico in enumerate(picos, 1):
                    inicio, fim, motivo = estimar_limites(filtrado, pico, fs)
                    if inicio is None or fim is None or not (0 <= inicio < pico < fim < len(sinal)):
                        registrar(batimento=numero, pico=pico, motivo=motivo)
                        continue
                    duracao = (fim - inicio) / fs * 1000
                    alargado = bool(duracao >= LIMIAR_QRS_MS)
                    for deslocamento, amplitude in enumerate(sinal[inicio:fim + 1]):
                        linhas.append({'ecg_id': item['id'], 'derivacao': derivacao,
                                      'batimento': numero, 'amostra_qrs': deslocamento,
                                      'amostra_original': inicio + deslocamento,
                                      'tempo_relativo_ms': deslocamento / fs * 1000,
                                      'amplitude_mV': amplitude, 'duracao_qrs_ms': duracao,
                                      'qrs_alargado': alargado})
                    registrar(numero, pico, inicio, fim, round(duracao, 2), alargado,
                              'ESTIMATIVA_EXPLORATORIA', 'REQUER_VALIDACAO')
                    total += 1
        por_derivacao[derivacao] = linhas

    # Montagem em diretório temporário: só publica pasta completa.
    destino.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='.qrs_tmp_', dir=destino.parent) as tmp:
        temp = Path(tmp)
        shutil.copy2(item['dat'], temp / item['dat'].name)
        shutil.copy2(item['hea'], temp / item['hea'].name)
        for derivacao in DERIVACOES:
            pd.DataFrame(por_derivacao[derivacao], columns=COLUNAS_QRS).to_csv(
                temp / f'{derivacao}_QRS.csv', index=False)
        pd.DataFrame(metadados, columns=COLUNAS_META).to_csv(temp / 'instancias.csv', index=False)
        if destino.exists():
            if not SOBRESCREVER:
                return {'ecg_id': item['id'], 'classe': item['classe'],
                        'situacao': 'DESTINO_JA_EXISTE', 'qrs_extraidos': 0}
            shutil.rmtree(destino)  # apenas pasta de saída da amostra
        shutil.move(str(temp), str(destino))
    return {'ecg_id': item['id'], 'classe': item['classe'],
            'situacao': 'PROCESSADO', 'qrs_extraidos': total}

# ============================================================
# 7. EXECUTAR TODAS AS CLASSES, SEM AMOSTRAGEM
# ============================================================
DESTINO.mkdir(parents=True, exist_ok=True)
resumo = []
for item in tqdm(registros, desc='Extração QRS — todas as amostras'):
    try:
        resumo.append(processar_amostra(item))
    except Exception as exc:
        resumo.append({'ecg_id': item['id'], 'classe': item['classe'],
                       'situacao': 'ERRO', 'qrs_extraidos': 0, 'erro': str(exc)})
        print(f"Erro: classe {item['classe']}, ECG {item['id']}: {exc}")

# ============================================================
# 8. RESUMO POR CLASSE
# ============================================================
df_resumo = pd.DataFrame(resumo)
if not df_resumo.empty:
    tabela = (df_resumo.groupby(['classe', 'situacao'], as_index=False)
              .agg(amostras=('ecg_id', 'count'), qrs_extraidos=('qrs_extraidos', 'sum')))
    print('\nRESUMO POR CLASSE E SITUAÇÃO:')
    print(tabela.to_string(index=False))
    df_resumo.to_csv(DESTINO / 'resumo_processamento.csv', index=False)
print('\nECG_Organizados preservada:', ORIGEM)
print('Recortes salvos em:', DESTINO)


Origem (somente leitura): c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\ECG_Organizados
Destino: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\Recortes-QRS
Registros elegíveis: 39
Ignorados: {'excluidos': 0, 'sem_dat': 0, 'fora_estrutura': 0, 'nome_invalido': 36515}


Extração QRS — todas as amostras: 100%|██████████| 39/39 [00:02<00:00, 18.82it/s]



RESUMO POR CLASSE E SITUAÇÃO:
    classe   situacao  amostras  qrs_extraidos
         0 PROCESSADO         8            820
    1.1_MI PROCESSADO         4            504
  1.2_STTC PROCESSADO         5            599
    1.3_CD PROCESSADO         4            484
   1.4_HYP PROCESSADO         3            338
  1.5_FORM PROCESSADO        10           1220
1.6_RHYTHM PROCESSADO         5            660

ECG_Organizados preservada: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\ECG_Organizados
Recortes salvos em: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\Recortes-QRS


# Extração Automatizada dos Complexos QRS nas 12 Derivações e Organização dos Recortes por Classe *geral*

In [21]:
import re
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, sosfiltfilt, find_peaks
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURAÇÕES: TODAS AS AMOSTRAS
# ============================================================
PASTA = Path.cwd()
DESTINO = PASTA / 'Recortes-QRS'
DERIVACOES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF',
              'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
CLASSES = ['0', '1.1_MI', '1.2_STTC', '1.3_CD',
           '1.4_HYP', '1.5_FORM', '1.6_RHYTHM']
LIMIAR_QRS_MS = 120
SOBRESCREVER = False  # True apenas para reprocessar pastas completas
JANELA_ANTES_MS = 120
JANELA_DEPOIS_MS = 160

# ============================================================
# 2. ENCONTRAR ORIGEM SEM MODIFICÁ-LA
# ============================================================
candidatas = [p for p in PASTA.rglob('*')
              if p.is_dir() and p.name.casefold() == 'ecg_organizados'
              and DESTINO.resolve() not in p.resolve().parents]
if len(candidatas) != 1:
    raise RuntimeError(f'Esperada uma pasta ECG_Organizados; encontradas: {candidatas}')
ORIGEM = candidatas[0]
if ORIGEM.resolve() == DESTINO.resolve() or ORIGEM.resolve() in DESTINO.resolve().parents:
    raise RuntimeError('Origem e destino não podem coincidir ou se sobrepor.')

# ============================================================
# 3. EXCLUSÃO DAS 445 INSTÂNCIAS
# ============================================================
arquivos_exclusao = sorted((PASTA / 'resultados').glob('instancias_NORM_e_anormal_juntos*.csv'))
if len(arquivos_exclusao) != 1:
    raise RuntimeError('É necessário exatamente um CSV de exclusões em resultados/.')
exclusoes = pd.read_csv(arquivos_exclusao[0])
if 'ecg_id' not in exclusoes:
    raise ValueError('CSV de exclusões sem coluna ecg_id.')
ids_excluidos = set(pd.to_numeric(exclusoes.ecg_id, errors='raise').astype(int))
if len(ids_excluidos) != 445:
    raise ValueError(f'Esperados 445 IDs únicos excluídos; encontrados {len(ids_excluidos)}.')

# ============================================================
# 4. DETECÇÃO EXPLORATÓRIA DE QRS
# Se as funções já foram definidas no notebook, preservá-las.
# ============================================================
if not callable(globals().get('detectar_qrs')):
    def detectar_qrs(sinal, fs):
        sinal = np.asarray(sinal, dtype=float)
        if len(sinal) < 30 or not np.isfinite(sinal).all():
            return [], None, 'SINAL_INVALIDO'
        filtrado = sosfiltfilt(butter(3, [5, 25], btype='bandpass', fs=fs, output='sos'), sinal)
        energia = filtrado ** 2
        janela = max(1, int(.08 * fs))
        energia_suave = np.convolve(energia, np.ones(janela) / janela, mode='same')
        limiar = np.median(energia_suave) + .5 * np.std(energia_suave)
        candidatos, _ = find_peaks(
            energia_suave, distance=max(1, int(.25 * fs)), height=limiar,
            prominence=max(np.std(energia_suave) * .25, np.finfo(float).eps))
        picos = []
        for candidato in candidatos:
            margem = int(.06 * fs)
            a, b = max(0, candidato - margem), min(len(filtrado), candidato + margem + 1)
            if a < b:
                picos.append(int(a + np.argmax(np.abs(filtrado[a:b]))))
        selecionados = []
        for pico in sorted(set(picos)):
            if not selecionados or pico - selecionados[-1] >= int(.20 * fs):
                selecionados.append(pico)
            elif abs(filtrado[pico]) > abs(filtrado[selecionados[-1]]):
                selecionados[-1] = pico
        return selecionados, filtrado, ('DETECCAO_PRELIMINAR' if selecionados else 'PICO_R_NAO_DETECTADO')

if not callable(globals().get('estimar_limites')):
    def estimar_limites(sinal_filtrado, pico, fs):
        antes, depois = int(JANELA_ANTES_MS * fs / 1000), int(JANELA_DEPOIS_MS * fs / 1000)
        a, b = max(0, pico - antes), min(len(sinal_filtrado) - 1, pico + depois)
        if a == 0 or b == len(sinal_filtrado) - 1:
            return None, None, 'QRS_PROXIMO_DA_BORDA'
        amplitude = np.abs(sinal_filtrado)
        referencia = amplitude[pico]
        if referencia <= 0:
            return None, None, 'AMPLITUDE_INSUFICIENTE'
        limiar = .10 * referencia
        inicio = fim = int(pico)
        while inicio > a and amplitude[inicio] > limiar:
            inicio -= 1
        while fim < b and amplitude[fim] > limiar:
            fim += 1
        if inicio >= pico or fim <= pico:
            return None, None, 'LIMITES_INDEFINIDOS'
        if inicio == a or fim == b:
            return None, None, 'LIMITES_NAO_CONFIAVEIS'
        return inicio, fim, 'LIMITES_ESTIMADOS'

# ============================================================
# 5. IDENTIFICAR AMOSTRAS E PRESERVAR CAMINHOS ORIGINAIS
# ============================================================
def classe_do_caminho(rel):
    partes = rel.parts
    if len(partes) >= 2 and partes[0] == '0' and partes[1].startswith('registros'):
        return '0'
    if len(partes) >= 3 and partes[0] == '1' and partes[1] in CLASSES[1:] and partes[2].startswith('registros'):
        return partes[1]
    return None

# Cada arquivo .hea identifica um ECG, mesmo quando não há pasta individual.
# Se houver _hr e _lr para o mesmo ECG e classe, preferir _hr (500 Hz).
selecionados = {}
ignorados = {'excluidos': 0, 'sem_dat': 0, 'fora_estrutura': 0,
            'nome_invalido': 0, 'versao_alternativa': 0}
log_selecao = []

for hea in sorted(ORIGEM.rglob('*.hea')):
    rel_pasta = hea.parent.relative_to(ORIGEM)
    classe = classe_do_caminho(rel_pasta)
    if classe is None:
        ignorados['fora_estrutura'] += 1
        log_selecao.append({'arquivo': str(hea.relative_to(ORIGEM)), 'situacao': 'FORA_ESTRUTURA'})
        continue

    m = re.fullmatch(r'(\d+)(?:_(hr|lr))?', hea.stem, flags=re.IGNORECASE)
    if m is None:
        ignorados['nome_invalido'] += 1
        log_selecao.append({'arquivo': str(hea.relative_to(ORIGEM)), 'situacao': 'NOME_INVALIDO'})
        continue

    ecg_id = int(m.group(1))
    if ecg_id in ids_excluidos:
        ignorados['excluidos'] += 1
        log_selecao.append({'arquivo': str(hea.relative_to(ORIGEM)), 'ecg_id': ecg_id, 'situacao': 'EXCLUIDO_445'})
        continue

    dat = hea.with_suffix('.dat')
    if not dat.is_file():
        ignorados['sem_dat'] += 1
        log_selecao.append({'arquivo': str(hea.relative_to(ORIGEM)), 'ecg_id': ecg_id, 'situacao': 'SEM_DAT'})
        continue

    # Mantém 0/registrosXXXX/ID ou 1/classe/registrosXXXX/ID.
    # Se o diretório do ECG já existe, reutiliza-o.
    if rel_pasta.name.isdigit() and int(rel_pasta.name) == ecg_id:
        rel_destino = rel_pasta
    else:
        rel_destino = rel_pasta / str(ecg_id).zfill(max(5, len(m.group(1))))

    item = {'id': ecg_id, 'classe': classe, 'hea': hea,
            'dat': dat, 'rel': rel_destino, 'versao': (m.group(2) or '').lower()}
    chave = str(rel_destino).casefold()
    atual = selecionados.get(chave)
    prioridade = {'hr': 2, '': 1, 'lr': 0}
    if atual is None:
        selecionados[chave] = item
    elif prioridade[item['versao']] > prioridade[atual['versao']]:
        ignorados['versao_alternativa'] += 1
        log_selecao.append({'arquivo': str(atual['hea'].relative_to(ORIGEM)),
                            'ecg_id': ecg_id, 'situacao': 'VERSAO_ALTERNATIVA'})
        selecionados[chave] = item
    else:
        ignorados['versao_alternativa'] += 1
        log_selecao.append({'arquivo': str(hea.relative_to(ORIGEM)),
                            'ecg_id': ecg_id, 'situacao': 'VERSAO_ALTERNATIVA'})

registros = sorted(selecionados.values(), key=lambda x: str(x['rel']))
for item in registros:
    log_selecao.append({'arquivo': str(item['hea'].relative_to(ORIGEM)),
                        'ecg_id': item['id'], 'classe': item['classe'],
                        'situacao': 'ELEGIVEL', 'versao': item['versao']})

print('Origem (somente leitura):', ORIGEM)
print('Destino:', DESTINO)
print('Registros elegíveis:', len(registros))
print('Ignorados:', ignorados)
print('IDs da lista de exclusão:', len(ids_excluidos))
if not registros:
    raise RuntimeError('Nenhuma amostra elegível encontrada na estrutura esperada.')

# ============================================================
# 6. PROCESSAR E SALVAR CADA AMOSTRA INDIVIDUALMENTE
# ============================================================
COLUNAS_QRS = ['ecg_id', 'derivacao', 'batimento', 'amostra_qrs',
               'amostra_original', 'tempo_relativo_ms', 'amplitude_mV',
               'duracao_qrs_ms', 'qrs_alargado']
COLUNAS_META = ['ecg_id', 'classe', 'derivacao', 'batimento', 'pico_r',
                'inicio_qrs', 'fim_qrs', 'duracao_ms', 'qrs_alargado',
                'situacao', 'motivo', 'fs_hz']

def processar_amostra(item):
    destino = DESTINO / item['rel']
    esperados = [item['dat'].name, item['hea'].name, 'instancias.csv'] + [f'{d}_QRS.csv' for d in DERIVACOES]
    if destino.exists() and not SOBRESCREVER:
        completos = all((destino / nome).is_file() for nome in esperados)
        return {'ecg_id': item['id'], 'classe': item['classe'],
                'situacao': 'JA_PROCESSADO' if completos else 'DESTINO_INCOMPLETO', 'qrs_extraidos': 0}

    registro = wfdb.rdrecord(str(item['hea'].with_suffix('')))
    fs = float(registro.fs)
    if registro.p_signal is None or fs <= 0:
        raise ValueError('Sinal físico ausente ou frequência inválida.')
    metadados = []
    por_derivacao = {}
    total = 0
    for derivacao in DERIVACOES:
        linhas = []
        def registrar(batimento=None, pico=None, inicio=None, fim=None,
                     duracao=None, alargado=None, situacao='REVISAR', motivo=''):
            metadados.append({'ecg_id': item['id'], 'classe': item['classe'],
                              'derivacao': derivacao, 'batimento': batimento,
                              'pico_r': pico, 'inicio_qrs': inicio, 'fim_qrs': fim,
                              'duracao_ms': duracao, 'qrs_alargado': alargado,
                              'situacao': situacao, 'motivo': motivo, 'fs_hz': fs})
        if derivacao not in registro.sig_name:
            registrar(motivo='DERIVACAO_AUSENTE')
        else:
            sinal = registro.p_signal[:, registro.sig_name.index(derivacao)]
            picos, filtrado, status = detectar_qrs(sinal, fs)
            if not picos:
                registrar(motivo=status)
            else:
                for numero, pico in enumerate(picos, 1):
                    inicio, fim, motivo = estimar_limites(filtrado, pico, fs)
                    if inicio is None or fim is None or not (0 <= inicio < pico < fim < len(sinal)):
                        registrar(batimento=numero, pico=pico, motivo=motivo)
                        continue
                    duracao = (fim - inicio) / fs * 1000
                    alargado = bool(duracao >= LIMIAR_QRS_MS)
                    for deslocamento, amplitude in enumerate(sinal[inicio:fim + 1]):
                        linhas.append({'ecg_id': item['id'], 'derivacao': derivacao,
                                      'batimento': numero, 'amostra_qrs': deslocamento,
                                      'amostra_original': inicio + deslocamento,
                                      'tempo_relativo_ms': deslocamento / fs * 1000,
                                      'amplitude_mV': amplitude, 'duracao_qrs_ms': duracao,
                                      'qrs_alargado': alargado})
                    registrar(numero, pico, inicio, fim, round(duracao, 2), alargado,
                              'ESTIMATIVA_EXPLORATORIA', 'REQUER_VALIDACAO')
                    total += 1
        por_derivacao[derivacao] = linhas

    # Montagem em diretório temporário: só publica pasta completa.
    destino.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='.qrs_tmp_', dir=destino.parent) as tmp:
        temp = Path(tmp)
        shutil.copy2(item['dat'], temp / item['dat'].name)
        shutil.copy2(item['hea'], temp / item['hea'].name)
        for derivacao in DERIVACOES:
            pd.DataFrame(por_derivacao[derivacao], columns=COLUNAS_QRS).to_csv(
                temp / f'{derivacao}_QRS.csv', index=False)
        pd.DataFrame(metadados, columns=COLUNAS_META).to_csv(temp / 'instancias.csv', index=False)
        if destino.exists():
            if not SOBRESCREVER:
                return {'ecg_id': item['id'], 'classe': item['classe'],
                        'situacao': 'DESTINO_JA_EXISTE', 'qrs_extraidos': 0}
            shutil.rmtree(destino)  # apenas pasta de saída da amostra
        shutil.move(str(temp), str(destino))
    return {'ecg_id': item['id'], 'classe': item['classe'],
            'situacao': 'PROCESSADO', 'qrs_extraidos': total}

# ============================================================
# 7. EXECUTAR TODAS AS CLASSES, SEM AMOSTRAGEM
# ============================================================
DESTINO.mkdir(parents=True, exist_ok=True)
pd.DataFrame(log_selecao).to_csv(DESTINO / 'auditoria_selecao.csv', index=False)
resumo = []
for item in tqdm(registros, desc='Extração QRS — todas as amostras'):
    try:
        resumo.append(processar_amostra(item))
    except Exception as exc:
        resumo.append({'ecg_id': item['id'], 'classe': item['classe'],
                       'situacao': 'ERRO', 'qrs_extraidos': 0, 'erro': str(exc)})
        print(f"Erro: classe {item['classe']}, ECG {item['id']}: {exc}")

# ============================================================
# 8. RESUMO POR CLASSE
# ============================================================
df_resumo = pd.DataFrame(resumo)
if not df_resumo.empty:
    tabela = (df_resumo.groupby(['classe', 'situacao'], as_index=False)
              .agg(amostras=('ecg_id', 'count'), qrs_extraidos=('qrs_extraidos', 'sum')))
    print('\nRESUMO POR CLASSE E SITUAÇÃO:')
    print(tabela.to_string(index=False))
    df_resumo.to_csv(DESTINO / 'resumo_processamento.csv', index=False)
print('\nECG_Organizados preservada:', ORIGEM)
print('Recortes salvos em:', DESTINO)
print('Auditoria da seleção:', DESTINO / 'auditoria_selecao.csv')


Origem (somente leitura): c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\ECG_Organizados
Destino: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\Recortes-QRS
Registros elegíveis: 36554
Ignorados: {'excluidos': 0, 'sem_dat': 0, 'fora_estrutura': 0, 'nome_invalido': 0, 'versao_alternativa': 0}
IDs da lista de exclusão: 445


Extração QRS — todas as amostras: 100%|██████████| 36554/36554 [32:50<00:00, 18.55it/s]  



RESUMO POR CLASSE E SITUAÇÃO:
    classe      situacao  amostras  qrs_extraidos
         0 JA_PROCESSADO         8              0
         0    PROCESSADO      7249         748671
    1.1_MI JA_PROCESSADO         4              0
    1.1_MI    PROCESSADO      5464         612342
  1.2_STTC JA_PROCESSADO         5              0
  1.2_STTC    PROCESSADO      5197         597621
    1.3_CD JA_PROCESSADO         4              0
    1.3_CD    PROCESSADO      4479         505684
   1.4_HYP JA_PROCESSADO         3              0
   1.4_HYP    PROCESSADO      2641         297423
  1.5_FORM JA_PROCESSADO        10              0
  1.5_FORM    PROCESSADO      7223         802542
1.6_RHYTHM JA_PROCESSADO         5              0
1.6_RHYTHM    PROCESSADO      4262         520011

ECG_Organizados preservada: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\resultados\ECG_Organizados
Recortes salvos em: c:\Users\Matheus\Documents\Detctor-automatizado-QRS\Recortes-QRS
Auditoria da seleção: c:\U

# Análise de Completude e Integridade dos Recortes QRS por Derivação

In [22]:
from pathlib import Path
import pandas as pd

PASTA_QRS = Path.cwd() / "Recortes-QRS"

DERIVACOES = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6"
]

resultados = []

for arquivo in PASTA_QRS.rglob("instancias.csv"):
    pasta_ecg = arquivo.parent
    dados = pd.read_csv(arquivo)

    if dados.empty:
        continue

    ecg_id = pasta_ecg.name
    classe = next(
        (parte for parte in pasta_ecg.parts
         if parte in ["0", "1.1_MI", "1.2_STTC",
                      "1.3_CD", "1.4_HYP",
                      "1.5_FORM", "1.6_RHYTHM"]),
        "DESCONHECIDA"
    )

    for derivacao in DERIVACOES:
        linhas = dados[dados["derivacao"] == derivacao]

        candidatos = linhas["batimento"].notna().sum()
        validos = (linhas["situacao"] == "ESTIMATIVA_EXPLORATORIA").sum()
        falhas = candidatos - validos

        arquivo_recorte = pasta_ecg / f"{derivacao}_QRS.csv"
        existe_csv = arquivo_recorte.is_file()

        if candidatos > 0:
            percentual = 100 * validos / candidatos
        else:
            percentual = 0.0

        if not existe_csv:
            situacao = "CSV AUSENTE"
        elif linhas["motivo"].eq("DERIVACAO_AUSENTE").any():
            situacao = "DERIVAÇÃO AUSENTE"
        elif candidatos == 0:
            situacao = "SEM QRS DETECTADO"
        elif validos == candidatos:
            situacao = "COMPLETA"
        elif validos > 0:
            situacao = "PARCIAL"
        else:
            situacao = "SEM RECORTES VÁLIDOS"

        resultados.append({
            "ecg_id": ecg_id,
            "classe": classe,
            "derivacao": derivacao,
            "qrs_candidatos": candidatos,
            "qrs_validos": validos,
            "qrs_falhas": falhas,
            "aproveitamento_%": round(percentual, 2),
            "situacao": situacao
        })

relatorio = pd.DataFrame(resultados)

resumo = (
    relatorio.groupby(["derivacao", "situacao"])
    .size()
    .unstack(fill_value=0)
    .reindex(DERIVACOES)
)

print("RESUMO DE COMPLETUDE POR DERIVAÇÃO")
display(resumo)

print("\nRELATÓRIO INDIVIDUAL")
display(relatorio)

relatorio.to_csv(
    PASTA_QRS / "relatorio_completude_qrs.csv",
    index=False
)

resumo.to_csv(
    PASTA_QRS / "resumo_completude_derivacoes.csv"
)

RESUMO DE COMPLETUDE POR DERIVAÇÃO


situacao,COMPLETA,DERIVAÇÃO AUSENTE,PARCIAL,SEM QRS DETECTADO,SEM RECORTES VÁLIDOS
derivacao,,,,,
I,28403,0,8150,0,1
II,28505,0,8049,0,0
III,28376,0,8177,0,1
aVR,0,36554,0,0,0
aVL,0,36554,0,0,0
aVF,0,36554,0,0,0
V1,28475,0,8079,0,0
V2,28506,0,8048,0,0
V3,28440,0,8110,0,4



RELATÓRIO INDIVIDUAL


,ecg_id,classe,derivacao,qrs_candidatos,qrs_validos,qrs_falhas,aproveitamento_%,situacao
0,00003,0,I,11,10,1,90.91,PARCIAL
1,00003,0,II,11,10,1,90.91,PARCIAL
2,00003,0,III,13,12,1,92.31,PARCIAL
3,00003,0,aVR,0,0,0,0.00,DERIVAÇÃO AUSENTE
4,00003,0,aVL,0,0,0,0.00,DERIVAÇÃO AUSENTE
...,...,...,...,...,...,...,...,...
438643,21833,1.6_RHYTHM,V2,18,18,0,100.00,COMPLETA
438644,21833,1.6_RHYTHM,V3,18,18,0,100.00,COMPLETA
438645,21833,1.6_RHYTHM,V4,18,18,0,100.00,COMPLETA
438646,21833,1.6_RHYTHM,V5,18,18,0,100.00,COMPLETA
